In [1]:

import pandas as pd
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

client = bigquery.Client(project='ecommerce-sql-analysis')

# 1. Define your dataset and table IDs
dataset_id = "ecommerce-sql-analysis.ecommerce_data"
table_id = f"{dataset_id}.all_sessions_cleaned"

# 2. Check if the dataset exists, if not, create it
try:
    client.get_dataset(dataset_id)
    print("Dataset already exists.")
except NotFound:
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"  # Match the source data location
    client.create_dataset(dataset)
    print(f"Created dataset {dataset_id}")

# 3. Now run your query logic (same as before)
sql_query = "SELECT DISTINCT * FROM `data-to-insights.ecommerce.all_sessions_raw`"
job_config = bigquery.QueryJobConfig(destination=table_id)
job_config.write_disposition = "WRITE_TRUNCATE"

query_job = client.query(sql_query, job_config=job_config)
query_job.result() 

print(f"Success! Table created: {table_id}")

Dataset already exists.
Success! Table created: ecommerce-sql-analysis.ecommerce_data.all_sessions_cleaned


In [ ]:
query_raw_dupes = """ 
SELECT COUNT(*) AS num_duplicate_rows

FROM `data-to-insights.ecommerce.all_sessions_raw`

GROUP BY fullVisitorId, channelGrouping, time, country, city,
totalTransactionRevenue, transactions, timeOnSite, pageviews,
sessionQualityDim, date, visitId, type, productRefundAmount,
productQuantity, productPrice, productRevenue, productSKU,
v2ProductName, v2ProductCategory, productVariant, currencyCode,
itemQuantity, itemRevenue, transactionRevenue, transactionId,
pageTitle, searchKeyword, pagePathLevel1, eCommerceAction_type,
eCommerceAction_step, eCommerceAction_option
HAVING num_duplicate_rows > 1

"""

df = client.query(query_raw_dupes).to_dataframe()
df.head()

In [ ]:
query_raw_count = """ 
SELECT 
fullVisitorId,
visitId,
date,
time,
v2ProductName,
productSKU,
type,
eCommerceAction_type,
eCommerceAction_step,
eCommerceAction_option,
transactionRevenue,
transactionId,

COUNT(*) AS row_count

FROM `data-to-insights.ecommerce.all_sessions`

GROUP BY  1,2,3,4,5,6,7,8,9,10,11,12

HAVING row_count > 1

"""

df = client.query(query_raw_count).to_dataframe()
df.head()

In [ ]:
query_total_views = """ 
SELECT 

COUNT(*) AS pageviews,
CouNT(DISTINCT fullVisitorId) AS users

FROM `data-to-insights.ecommerce.all_sessions`



"""

df = client.query(query_total_views).to_dataframe()
df.head()

In [ ]:
query_by_channel = """ 
SELECT 
channelGrouping,
COUNT(*) AS pageviews,
CouNT(DISTINCT fullVisitorId) AS users

FROM `data-to-insights.ecommerce.all_sessions`
GROUP BY channelGrouping


"""

df = client.query(query_by_channel).to_dataframe()
df.head()

In [23]:
query_product_names = """ 
SELECT 
v2ProductName

FROM `data-to-insights.ecommerce.all_sessions`
GROUP BY v2ProductName
ORDER BY v2ProductName
"""

df = client.query(query_product_names).to_dataframe()
len(df)

c:\Users\Naveen\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


633

In [6]:
query_top5_products = """ 
SELECT 
v2ProductName,
COUNT(*) AS product_views

FROM `data-to-insights.ecommerce.all_sessions`
WHERE TYPE = 'PAGE'
GROUP BY v2ProductName
ORDER BY product_views DESC
LIMIT 5
"""

df = client.query(query_top5_products).to_dataframe()
df

c:\Users\Naveen\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,v2ProductName,product_views
0,Google Men's 100% Cotton Short Sleeve Hero Tee...,316482
1,22 oz YouTube Bottle Infuser,221558
2,YouTube Men's Short Sleeve Hero Tee Black,210700
3,Google Men's 100% Cotton Short Sleeve Hero Tee...,202205
4,YouTube Custom Decals,200789


In [14]:
query_orders_units = """ 

SELECT 
v2ProductName,

COUNT(productQuantity) AS orders,
COUNT(*) AS product_views,
SUM(productQuantity) AS total_units,
ROUND(SUM(productQuantity) / COUNT(productQuantity), 2) AS avg_per_order

FROM `data-to-insights.ecommerce.all_sessions`
WHERE type ='PAGE'
GROUP BY v2ProductName
ORDER BY product_views DESC
LIMIT 5

"""

df = client.query(query_orders_units).to_dataframe()
df

c:\Users\Naveen\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,v2ProductName,orders,product_views,total_units,avg_per_order
0,Google Men's 100% Cotton Short Sleeve Hero Tee...,3158,316482,6352,2.01
1,22 oz YouTube Bottle Infuser,508,221558,4769,9.39
2,YouTube Men's Short Sleeve Hero Tee Black,949,210700,1114,1.17
3,Google Men's 100% Cotton Short Sleeve Hero Tee...,2713,202205,8072,2.98
4,YouTube Custom Decals,1703,200789,11336,6.66


In [ ]:
query_orders_units = """ 

SELECT
v2ProductName,
  COUNT(*) AS product_views,
  COUNT(productQuantity) AS potential_orders,
  SUM(productQuantity) AS quantity_product_added,
  
  ROUND(COUNT(productQuantity) / COUNT(*)*100.0, 2) AS conversion_rate
FROM `data-to-insights.ecommerce.all_sessions`
WHERE Lower(v2ProductName) NOT LIKE '%frisbee%'
GROUP BY v2ProductName
HAVING quantity_product_added > 1000
ORDER BY conversion_rate DESC
LIMIT 10;

"""

df = client.query(query_orders_units).to_dataframe()
df

In [ ]:
# Challenge 3: Track abandoned carts from high quality sessions.
# In plain English: find visitors who added something to their 
# cart but never completed checkout — and only look at high quality sessions.

query_orders_units = """ 
SELECT
CONCAT(FullVisitorId, '-', CAST(visitId AS STRING)) AS unique_session_id,
sessionQualityDim,
MAX(eCommerceAction_type) AS checkout_progress
FROM `data-to-insights.ecommerce.all_sessions`
WHERE sessionQualityDim > 60
GROUP BY unique_session_id, sessionQualityDim
HAVING 
    checkout_progress ='3'
    AND (SUM(productRevenue) = 0 OR SUM(transactionRevenue) IS NULL)
"""

df = client.query(query_orders_units).to_dataframe()
df